### Consolidating Basic HDB Data

In this notebook, we will be consolidating basic HDB data into one file. We will then be adding supplementary property information to the basic HDB resale dataframe. 

Data Source: Various data sets on HDB resale flat prices (https://beta.data.gov.sg/)

- Resale Flat Prices 1990 - 1999
- Resale Flat Prices 2000 - Feb 2012
- Resale Flat Prices Mar 2012 to Dec 2014
- Resale Flat Prices Jan 2015 to Dec 2016
- Resale flat prices Jan-2017 onwards
- HDB Property Information

### Setup

In [1]:
%pip install -q numpy pandas matplotlib seaborn scikit-learn requests


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Load libraries

In [1]:
# Import all relevant libraries 

# Wrangling libraries
import pandas as pd
import numpy as np


# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Os Libraries
import os

# Other libraries 
import zipfile
import requests
import time
from datetime import datetime

In [2]:
# Configurations
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x :.3f}")
sns.set_theme(style="darkgrid")

RANDOM_STATE = 100
TARGET_COL = "resale_price" # target column name

### Load Data

In [3]:
# Navigate to project workspace
os.chdir('/workspaces/DSE3101-Project')

# Verify correct directory location 
print("Current directory:", os.getcwd())
print("Contents:", os.listdir('.'))

# Navigate to data/raw folder to obtain zip file that contains raw data
os.chdir('data/raw')
current_dir = os.getcwd()
print(f"Latest directory: {current_dir}")

Current directory: /workspaces/DSE3101-Project
Contents: ['docs', '.git', 'notebooks', '.DS_Store', 'models', 'frontend', 'requirements.txt', '.gitignore', '.vscode', 'backend', 'requirements_fixed.txt', 'data', 'onemap_all_themes_full.json', 'README.md', 'onemap_all_themes_raw.txt']
Latest directory: /workspaces/DSE3101-Project/data/raw


In [4]:
# Go up to repo root, then into data folder to retrive raw data from zip file
zip_path = os.path.join(current_dir, "ResaleFlatPrices.zip")
zip_path

'/workspaces/DSE3101-Project/data/raw/ResaleFlatPrices.zip'

In [5]:
# Loop through contents of zip file to retrieve all records and merge them
all_dfs = []

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
    
    for csv_file in csv_files:
        with zip_ref.open(csv_file) as f:
            df = pd.read_csv(f)
            all_dfs.append(df)

# Combine into one DataFrame
combined_df = pd.concat(all_dfs, ignore_index=True)

print(f"✓ Loaded {len(all_dfs)} CSV files")
print(f"✓ Total records: {len(combined_df):,}")

✓ Loaded 5 CSV files
✓ Total records: 970,969


In [6]:
# View df information
print(combined_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 970969 entries, 0 to 970968
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                970969 non-null  str    
 1   town                 970969 non-null  str    
 2   flat_type            970969 non-null  str    
 3   block                970969 non-null  str    
 4   street_name          970969 non-null  str    
 5   storey_range         970969 non-null  str    
 6   floor_area_sqm       970969 non-null  float64
 7   flat_model           970969 non-null  str    
 8   lease_commence_date  970969 non-null  int64  
 9   resale_price         970969 non-null  float64
 10  remaining_lease      261919 non-null  object 
dtypes: float64(2), int64(1), object(1), str(7)
memory usage: 81.5+ MB
None


In [7]:
# Convert month column which consists of month and year into a date type
# Create new column of column date and drop unnecessary column
combined_df['sold_year_month'] = pd.to_datetime(combined_df['month'])
combined_df = combined_df.drop(columns='month')

In [8]:
# Creating just a year sold column
combined_df['sold_year'] = combined_df['sold_year_month'].dt.strftime('%Y').astype('int')

In [9]:
# Rationale for filtering early data:
# 1. Prevents model contamination from outdated market regimes (e.g., pre-2015 HDB market structure).
# 2. Reduces bias from old data, ensuring the model fits current pricing trends accurately.
# 3. Improves computational efficiency by dropping irrelevant historical records.

combined_df = combined_df[combined_df['sold_year'] >= 2015]
combined_df = combined_df.reset_index(drop=True)

In [10]:
print("=" * 80)
print("INITIAL DATA INSPECTION")
print("=" * 80)

print(f"\nDataset shape: {combined_df.shape}")

# View 5 random rows of dataframe
combined_df.sample(5, random_state=RANDOM_STATE)

INITIAL DATA INSPECTION

Dataset shape: (261919, 12)


,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,resale_price,remaining_lease,sold_year_month,sold_year
58919,BUKIT MERAH,3 ROOM,62,TELOK BLANGAH HTS,07 TO 09,73.000,New Generation,1976,345000.000,57 years 08 months,2018-02-01,2018
216257,HOUGANG,3 ROOM,682,HOUGANG AVE 4,01 TO 03,64.000,Simplified,1989,400000.000,64 years 01 month,2024-07-01,2024
226482,SENGKANG,4 ROOM,327C,ANCHORVALE RD,10 TO 12,92.000,Premium Apartment,2015,670000.000,89 years 09 months,2024-09-01,2024
149105,BISHAN,5 ROOM,291,BISHAN ST 24,25 TO 27,121.000,Premium Apartment,1998,960000.000,75 years 08 months,2021-11-01,2021
202117,TOA PAYOH,4 ROOM,138C,LOR 1A TOA PAYOH,25 TO 27,91.000,DBSS,2012,907000.000,88 years,2023-07-01,2023


In [11]:
# View df column types
print(combined_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 261919 entries, 0 to 261918
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   town                 261919 non-null  str           
 1   flat_type            261919 non-null  str           
 2   block                261919 non-null  str           
 3   street_name          261919 non-null  str           
 4   storey_range         261919 non-null  str           
 5   floor_area_sqm       261919 non-null  float64       
 6   flat_model           261919 non-null  str           
 7   lease_commence_date  261919 non-null  int64         
 8   resale_price         261919 non-null  float64       
 9   remaining_lease      261919 non-null  object        
 10  sold_year_month      261919 non-null  datetime64[us]
 11  sold_year            261919 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(2), object(1), str(6)
memory usage: 24.0+ M

In [12]:
print("\n" + "=" * 80)
print("MISSING VALUE ANALYSIS")
print("=" * 80)

missing_summary = pd.DataFrame({
    'Missing_Count': combined_df.isna().sum(),
    'Percentage': (combined_df.isna().sum() / len(combined_df)) * 100,
})
print(missing_summary)


MISSING VALUE ANALYSIS
                     Missing_Count  Percentage
town                             0       0.000
flat_type                        0       0.000
block                            0       0.000
street_name                      0       0.000
storey_range                     0       0.000
floor_area_sqm                   0       0.000
flat_model                       0       0.000
lease_commence_date              0       0.000
resale_price                     0       0.000
remaining_lease                  0       0.000
sold_year_month                  0       0.000
sold_year                        0       0.000


***NOTE: From the out above, there are no null values, df is complete***

In [13]:
# Check the count of each data type in the remaining_lease column
type_counts = combined_df['remaining_lease'].apply(type).value_counts()
print("Data types present in the column:")
print(type_counts)

Data types present in the column:
remaining_lease
<class 'str'>    224766
<class 'int'>     37153
Name: count, dtype: int64


In [15]:
# Derive number of years of lease remaining at sold year as integer 
combined_df['remaining_lease'] = 99 - (combined_df['sold_year'] - combined_df['lease_commence_date'])

# View range of leases remaining 
np.sort(combined_df['remaining_lease'].unique())

# Oldest hdb has 39 years remaining.
# Youngest is 98
# Still acceptable as key collection date for a new HDB BTO flat can be before the official lease commencement date
# You cannot sell your HDB flat within 5 years of the key collection date

array([39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55,
       56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72,
       73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
       90, 91, 92, 93, 94, 95, 96, 97, 98])

In [16]:
num_cols = combined_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = combined_df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Target column:", TARGET_COL)
print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Target column: resale_price
Numerical columns: ['floor_area_sqm', 'lease_commence_date', 'resale_price', 'remaining_lease', 'sold_year']
Categorical columns: ['town', 'flat_type', 'block', 'street_name', 'storey_range', 'flat_model', 'sold_year_month']


In [17]:
print("\n" + "=" * 80)
print("DUPLICATE REMOVAL")
print("=" * 80)

initial_rows = len(combined_df)
duplicates = combined_df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

print(f"Duplicate rows found:")
print(combined_df[combined_df.duplicated(keep=False)].head())

if duplicates > 0:
    final_df = combined_df.drop_duplicates()
    print(f"✓ Removed {initial_rows - len(combined_df)} duplicate rows")
    print(f"New shape: {final_df.shape}")


DUPLICATE REMOVAL
Duplicate rows found: 419
Duplicate rows found:
                 town flat_type block         street_name storey_range  \
660   KALLANG/WHAMPOA    3 ROOM    57       GEYLANG BAHRU     16 TO 18   
661   KALLANG/WHAMPOA    3 ROOM    57       GEYLANG BAHRU     16 TO 18   
2165         TAMPINES    3 ROOM   403      TAMPINES ST 41     07 TO 09   
2166         TAMPINES    3 ROOM   403      TAMPINES ST 41     07 TO 09   
3895            BEDOK    4 ROOM   701  BEDOK RESERVOIR RD     10 TO 12   

      floor_area_sqm      flat_model  lease_commence_date  resale_price  \
660           65.000        Improved                 1974    315000.000   
661           65.000        Improved                 1974    315000.000   
2165          69.000        Improved                 1985    350000.000   
2166          69.000        Improved                 1985    350000.000   
3895          93.000  New Generation                 1980    400000.000   

      remaining_lease sold_year_month

In [19]:
# View value counts and number unique for each column to check for validity
for col in final_df.select_dtypes(include=object):
    print(f"Number of unique values in {col}: {final_df[col].nunique()}")
    print(f"{final_df[col].value_counts()} \n")

/tmp/ipykernel_12407/2140683191.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in final_df.select_dtypes(include=object):


Number of unique values in town: 26
town
SENGKANG           20934
WOODLANDS          18535
TAMPINES           17997
PUNGGOL            17846
JURONG WEST        17676
YISHUN             17446
BEDOK              14120
HOUGANG            13242
CHOA CHU KANG      11902
ANG MO KIO         10976
BUKIT BATOK        10846
BUKIT MERAH         9918
BUKIT PANJANG       9413
TOA PAYOH           8397
KALLANG/WHAMPOA     8008
SEMBAWANG           7826
PASIR RIS           7653
QUEENSTOWN          7049
GEYLANG             6590
CLEMENTI            5921
JURONG EAST         5390
SERANGOON           4814
BISHAN              4604
CENTRAL AREA        2160
MARINE PARADE       1602
BUKIT TIMAH          635
Name: count, dtype: int64 

Number of unique values in flat_type: 7
flat_type
4 ROOM              110269
5 ROOM               63751
3 ROOM               63441
EXECUTIVE            18943
2 ROOM                4912
1 ROOM                  94
MULTI-GENERATION        90
Name: count, dtype: int64 

Number of uniq

In [20]:
# Flat model also seems to have many duplicates
# mapping logic - https://sg.finance.yahoo.com/news/different-types-hdb-houses-call-020000642.html
correction_map = {'2-ROOM':'2-room',
                  'APARTMENT':'Apartment',
                  'Improved-Maisonette':'Executive Maisonette',
                  'IMPROVED-MAISONETTE':'Executive Maisonette',
                  'IMPROVED':'Improved',
                  'MAISONETTE':'Maisonette',
                  'Model A-Maisonette':'Maisonette',
                  'MODEL A-MAISONETTE':'Maisonette',
                  'MODEL A':'Model A',
                  'MULTI GENERATION':'Multi Generation',
                  'Premium Apartment Loft':'Premium Apartment',
                  'PREMIUM APARTMENT':'Premium Apartment',
                  'Premium Maisonette':'Executive Maisonette',
                  'SIMPLIFIED':'Simplified',
                  'STANDARD':'Standard',
                  'TERRACE':'Terrace',
                  'NEW GENERATION':'New Generation'}

final_df = final_df.replace({'flat_model': correction_map})

final_df['flat_model'].value_counts()

flat_model
Model A                 91419
Improved                64319
New Generation          33759
Premium Apartment       28375
Simplified              10464
Apartment                9496
Maisonette               7670
Standard                 7185
DBSS                     3792
Model A2                 3115
Type S1                   500
Adjoined flat             429
2-room                    380
Type S2                   242
Terrace                   138
Multi Generation           90
3Gen                       70
Executive Maisonette       57
Name: count, dtype: int64

In [21]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 261500 entries, 0 to 261918
Data columns (total 12 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   town                 261500 non-null  str           
 1   flat_type            261500 non-null  str           
 2   block                261500 non-null  str           
 3   street_name          261500 non-null  str           
 4   storey_range         261500 non-null  str           
 5   floor_area_sqm       261500 non-null  float64       
 6   flat_model           261500 non-null  str           
 7   lease_commence_date  261500 non-null  int64         
 8   resale_price         261500 non-null  float64       
 9   remaining_lease      261500 non-null  int64         
 10  sold_year_month      261500 non-null  datetime64[us]
 11  sold_year            261500 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(3), str(6)
memory usage: 25.9 MB


### Load HDB Property Information Dataset

In [22]:
# Load and view Property Information data
df_info = pd.read_csv('HDBPropertyInformation.csv')
df_info.info()

<class 'pandas.DataFrame'>
RangeIndex: 13267 entries, 0 to 13266
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype
---  ------                 --------------  -----
 0   blk_no                 13267 non-null  str  
 1   street                 13267 non-null  str  
 2   max_floor_lvl          13267 non-null  int64
 3   year_completed         13267 non-null  int64
 4   residential            13267 non-null  str  
 5   commercial             13267 non-null  str  
 6   market_hawker          13267 non-null  str  
 7   miscellaneous          13267 non-null  str  
 8   multistorey_carpark    13267 non-null  str  
 9   precinct_pavilion      13267 non-null  str  
 10  bldg_contract_town     13267 non-null  str  
 11  total_dwelling_units   13267 non-null  int64
 12  1room_sold             13267 non-null  int64
 13  2room_sold             13267 non-null  int64
 14  3room_sold             13267 non-null  int64
 15  4room_sold             13267 non-null  int64
 1

### Joining the DataFrames on Addresses

In [23]:
# Make standardised address columns for both DFs
final_df['address'] = final_df['block'] + " " + final_df['street_name']
df_info['address'] = df_info['blk_no'] + " " + df_info['street']

In [24]:
# Do a left join to merge the two data frames into one 
# Since the resale df contains data of HDBs which do not exist anymore,
# use the df with property information for our left df 
# since it only contains information from HDBs which still exist.
df_full = final_df.merge(df_info, how='left', on='address')

# Transpose and view df as it is too wide to view
df_full.head().T

,0,1,2,3,4
town,ANG MO KIO,ANG MO KIO,ANG MO KIO,ANG MO KIO,ANG MO KIO
flat_type,3 ROOM,3 ROOM,3 ROOM,3 ROOM,3 ROOM
block,174,541,163,446,557
street_name,ANG MO KIO AVE 4,ANG MO KIO AVE 10,ANG MO KIO AVE 4,ANG MO KIO AVE 10,ANG MO KIO AVE 10
storey_range,07 TO 09,01 TO 03,01 TO 03,01 TO 03,07 TO 09
floor_area_sqm,60.000,68.000,69.000,68.000,68.000
flat_model,Improved,New Generation,New Generation,New Generation,New Generation
lease_commence_date,1986,1981,1980,1979,1980
resale_price,255000.000,275000.000,285000.000,290000.000,290000.000
remaining_lease,70,65,64,63,64


In [25]:
df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 261500 entries, 0 to 261499
Data columns (total 37 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   town                   261500 non-null  str           
 1   flat_type              261500 non-null  str           
 2   block                  261500 non-null  str           
 3   street_name            261500 non-null  str           
 4   storey_range           261500 non-null  str           
 5   floor_area_sqm         261500 non-null  float64       
 6   flat_model             261500 non-null  str           
 7   lease_commence_date    261500 non-null  int64         
 8   resale_price           261500 non-null  float64       
 9   remaining_lease        261500 non-null  int64         
 10  sold_year_month        261500 non-null  datetime64[us]
 11  sold_year              261500 non-null  int64         
 12  address                261500 non-null  str           


**Comments**

- There are some rows with no blk_no with no property information
- Since there is no need for them, we can remove them

In [26]:
# Count of how many rows have no blk_no
len(df_full[df_full['blk_no'].isna()]['address'].unique())

7

In [27]:
# Since only 7 rows of data are affected, we will remove them
df_full = df_full[df_full['blk_no'].notna()]

In [28]:
# remove repeated columns and rental information
df_full = df_full.drop(columns=['blk_no','street','1room_rental','2room_rental','3room_rental','other_room_rental','bldg_contract_town'])

In [29]:
# Data type conversion 
print("\n" + "=" * 80)
print("DATA TYPE CONVERSION & PARSING")
print("=" * 80)

# Ensure numeric columns are numeric
numeric_cols = ['floor_area', 'lease_commence', 'resale_price']
for col in numeric_cols:
    if col in df_full.columns:
        df_full[col] = pd.to_numeric(df_full[col], errors='coerce')


columns_to_int = ['max_floor_lvl', 'year_completed']
# Convert them to integers
df_full[columns_to_int] = df_full[columns_to_int].astype(int)

print("✓ Ensured numeric columns have correct data types")


DATA TYPE CONVERSION & PARSING
✓ Ensured numeric columns have correct data types


In [30]:
# Feature Engineering
print("\n" + "=" * 80)
print("FEATURE ENGINEERING")
print("=" * 80)

# Find middle of storey_range (e.g., Split "01 TO 05" → get both numbers → average → single storey_mid column)
split_storey = df_full['storey_range'].str.split(' TO ', expand=True).astype(int)
df_full['storey_mid'] = split_storey[[0,1]].mean(axis=1)


# Storey categories
df_full['storey_category'] = pd.cut(df_full['storey_mid'],
                                        bins=[0, 5, 10, 15, 50],
                                        labels=['Low', 'Mid-Low', 'Mid-High', 'High'])

print("✓ Created floor_area_category")

# Categorize towns into regions (CCR, RCR, OCR) based on information below:
# https://www.propertyguru.com.sg/property-guides/singapore-district-map-21045 

region_mapping = {
    # --- CCR (Core Central Region) ---
    'ORCHARD': 'CCR', 'SOMERSET': 'CCR', 'RIVER VALLEY': 'CCR',
    'TANGLIN': 'CCR', 'BUKIT TIMAH': 'CCR', 'HOLLAND': 'CCR',
    'NEWTON': 'CCR', 'NOVENA': 'CCR', 'DUNEARN': 'CCR', 'WATTEN': 'CCR',
    'BOAT QUAY': 'CCR', 'RAFFLES PLACE': 'CCR', 'MARINA DOWNTOWN': 'CCR', 'SUNTEC CITY': 'CCR',
    'SHENTON WAY': 'CCR', 'TANJONG PAGAR': 'CCR',
    'SENTOSA': 'CCR',
    'CITY HALL': 'CCR',
    'BUGIS': 'CCR',
    'CENTRAL AREA': 'CCR', # Official HDB Town

    # --- RCR (Rest of Central Region) ---
    'MARINA SOUTH': 'RCR',
    'CHINATOWN': 'RCR',
    'QUEENSTOWN': 'RCR', 'ALEXANDRA': 'RCR', 'TIONG BAHRU': 'RCR',
    'HARBOURFRONT': 'RCR', 'KEPPEL': 'RCR', 'TELOK BLANGAH': 'RCR',
    'BUONA VISTA': 'RCR', 'DOVER': 'RCR', 'PASIR PANJANG': 'RCR',
    'FORT CANNING': 'RCR',
    'ROCHOR': 'RCR',
    'LITTLE INDIA': 'RCR', 'FARRER PARK': 'RCR',
    'BALESTIER': 'RCR', 'WHAMPOA': 'RCR', 'TOA PAYOH': 'RCR', 'BOON KENG': 'RCR', 'BENDEMEER': 'RCR', 'KAMPONG BUGIS': 'RCR',
    'POTONG PASIR': 'RCR', 'BIDADARI': 'RCR', 'MACPHERSON': 'RCR', 'UPPER ALJUNIED': 'RCR',
    'GEYLANG': 'RCR', 'DAKOTA': 'RCR', 'PAYA LEBAR CENTRAL': 'RCR', 'EUNOS': 'RCR', 'UBI': 'RCR', 'ALJUNIED': 'RCR',
    'TANJONG RHU': 'RCR', 'AMBER': 'RCR', 'MEYER': 'RCR', 'KATONG': 'RCR', 'DUNMAN': 'RCR', 'JOO CHIAT': 'RCR', 'MARINE PARADE': 'RCR',
    'BISHAN': 'RCR', 'THOMSON': 'RCR',
    'BUKIT MERAH': 'RCR',      # Official HDB Town
    'KALLANG/WHAMPOA': 'RCR',  # Official HDB Town

    # --- OCR (Outside Central Region) ---
    'CLEMENTI': 'OCR', 'WEST COAST': 'OCR',
    'KEMBANGAN': 'OCR', 'KAKI BUKIT': 'OCR',
    'TELOK KURAU': 'OCR', 'SIGLAP': 'OCR', 'FRANKEL': 'OCR',
    'BEDOK': 'OCR', 'UPPER EAST COAST': 'OCR', 'BAYSHORE': 'OCR', 'TANAH MERAH': 'OCR', 'UPPER CHANGI': 'OCR',
    'FLORA DRIVE': 'OCR', 'LOYANG': 'OCR', 'CHANGI': 'OCR',
    'TAMPINES': 'OCR', 'PASIR RIS': 'OCR',
    'PUNGGOL': 'OCR', 'SENGKANG': 'OCR', 'HOUGANG': 'OCR', 'KOVAN': 'OCR', 'SERANGOON': 'OCR', 'LORONG AH SOO': 'OCR',
    'ANG MO KIO': 'OCR',
    'UPPER BUKIT TIMAH': 'OCR', 'ULU PANDAN': 'OCR', 'CLEMENTI PARK': 'OCR',
    'JURONG EAST': 'OCR', 'JURONG WEST': 'OCR', 'BOON LAY': 'OCR',
    'HILLVIEW': 'OCR', 'BUKIT PANJANG': 'OCR', 'BUKIT BATOK': 'OCR', 'CHOA CHU KANG': 'OCR',
    'KRANJI': 'OCR', 'LIM CHU KANG': 'OCR', 'SUNGEI GEDONG': 'OCR', 'TENGAH': 'OCR',
    'WOODLANDS': 'OCR', 'ADMIRALTY': 'OCR',
    'LENTOR': 'OCR', 'SPRINGLEAF': 'OCR', 'MANDAI': 'OCR',
    'YISHUN': 'OCR', 'SEMBAWANG': 'OCR',
    'SELETAR': 'OCR', 'SELETAR HILL': 'OCR', 'SENGKANG WEST': 'OCR'
}

# Apply the mapping. 
# Using .str.upper() ensures that even if your dataframe has 'Orchard' or 'orchard', it will match the uppercase keys.
df_full['region'] = df_full['town'].str.upper().map(region_mapping)
df_full['region'] = df_full['region'].fillna('Other')
print("✓ Created region feature")

# Flag mature vs non-mature estates
mature_estates = ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT MERAH', 'BUKIT TIMAH',
                  'CENTRAL AREA', 'CLEMENTI', 'GEYLANG', 'KALLANG/WHAMPOA',
                  'MARINE PARADE', 'PASIR RIS', 'QUEENSTOWN', 'SERANGOON', 'TAMPINES', 'TOA PAYOH']

df_full['is_mature_estate'] = df_full['town'].isin(mature_estates).astype(int)
print("✓ Created is_mature_estate flag")


FEATURE ENGINEERING
✓ Created floor_area_category
✓ Created region feature
✓ Created is_mature_estate flag


In [31]:
df_full['region'].value_counts()

region
OCR    212491
RCR     46160
CCR      2795
Name: count, dtype: int64

In [32]:
# Remove negative or zero prices
print("\n1. Validating resale_price...")
print(f"   Min price: ${df_full['resale_price'].min():,.0f}")
print(f"   Max price: ${df_full['resale_price'].max():,.0f}")

invalid_prices = df_full[df_full['resale_price'] <= 0]
print(f"   Found {len(invalid_prices)} records with price <= 0")
df_full = df_full[df_full['resale_price'] > 0]

# Rule 2: Floor area must be positive and realistic
print("\n2. Validating floor_area...")
print(f"   Min area: {df_full['floor_area_sqm'].min()} sqm")
print(f"   Max area: {df_full['floor_area_sqm'].max()} sqm")

invalid_area = df_full[df_full['floor_area_sqm'] <= 0]
print(f"   Found {len(invalid_area)} records with area <= 0")
df_full = df_full[df_full['floor_area_sqm'] > 0]


1. Validating resale_price...
   Min price: $140,000
   Max price: $1,658,888
   Found 0 records with price <= 0

2. Validating floor_area...
   Min area: 31.0 sqm
   Max area: 366.7 sqm
   Found 0 records with area <= 0


In [33]:
drop_cols = ['1room_sold','2room_sold','3room_sold','4room_sold','5room_sold','exec_sold','multigen_sold',
             'studio_apartment_sold','  1-Room Residential Properties','  2-Room Residential Properties',
             '  3-Room Residential Properties','  4-Room Residential Properties','  5-Room Residential Properties',
             '  Executive Properties', 'year_completed', 'residential', 'commercial', 'market_hawker', 'miscellaneous',
             'multistorey_carpark', 'precinct_pavilion', 'total_dwelling_units', 'sold_year_month']

# Drop the columns
hdb_df = df_full.drop(columns=drop_cols, errors='ignore')

In [34]:
hdb_df.info()

<class 'pandas.DataFrame'>
Index: 261446 entries, 0 to 261499
Data columns (total 17 columns):
 #   Column               Non-Null Count   Dtype   
---  ------               --------------   -----   
 0   town                 261446 non-null  str     
 1   flat_type            261446 non-null  str     
 2   block                261446 non-null  str     
 3   street_name          261446 non-null  str     
 4   storey_range         261446 non-null  str     
 5   floor_area_sqm       261446 non-null  float64 
 6   flat_model           261446 non-null  str     
 7   lease_commence_date  261446 non-null  int64   
 8   resale_price         261446 non-null  float64 
 9   remaining_lease      261446 non-null  int64   
 10  sold_year            261446 non-null  int64   
 11  address              261446 non-null  str     
 12  max_floor_lvl        261446 non-null  int64   
 13  storey_mid           261446 non-null  float64 
 14  storey_category      261446 non-null  category
 15  region          

In [35]:
final_duplicates = hdb_df.duplicated().sum()
print(f"Duplicate rows found: {final_duplicates}")

if final_duplicates > 0:
    hdb_df_final = hdb_df.drop_duplicates()

Duplicate rows found: 2303


### Get the nearest MRT, community club, clinics, parks & hawker centres. First we import the relevant libraries

In [43]:
import pandas as pd
import numpy as np
import geopandas as gpd
import requests
import time
import re
from pathlib import Path

### Set the necessary settings configurations

In [50]:
from pathlib import Path
cwd = Path.cwd()  # Returns Path object
print(cwd)  # e.g., Path('/home/user/project')

/workspaces/DSE3101-Project/data/raw


In [51]:
AMENITY_DIR = cwd
MRT_FILE = "LTAMRTStationExitGEOJSON.geojson"
CACHE_FILE = "address_coords_cache.csv"
OUTPUT_CSV = "final_df_with_location_features.csv"

RADIUS_M = 1000
GEOCODE_SLEEP = 0.25
SAVE_EVERY = 100

### Build our helper functions

In [52]:
def log(msg):
    print(f"[INFO] {msg}")

def find_file(folder, keywords):
    folder = Path(folder)
    files = list(folder.glob("*"))
    for f in files:
        name = f.name.lower()
        if all(k.lower() in name for k in keywords):
            return str(f)
    raise FileNotFoundError(f"Could not find file in {folder} matching keywords: {keywords}")

def normalize_to_wgs84(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    return gdf.to_crs(epsg=4326)

def project_to_sv21(gdf):
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    return gdf.to_crs(epsg=3414)

def build_points_gdf(df, lat_col="latitude", lon_col="longitude"):
    out = df.dropna(subset=[lat_col, lon_col]).copy()
    return gpd.GeoDataFrame(
        out,
        geometry=gpd.points_from_xy(out[lon_col], out[lat_col]),
        crs="EPSG:4326"
    )

def extract_from_description(desc, field_name):
    if pd.isna(desc):
        return None

    text = str(desc)
    patterns = [
        rf"{field_name}\s*</th>\s*<td[^>]*>\s*(.*?)\s*</td>",
        rf"{field_name}\s*</td>\s*<td[^>]*>\s*(.*?)\s*</td>",
        rf"{field_name}\s*[:\-]\s*(.*?)(?:<br|</td>|</tr>|$)",
    ]

    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE | re.DOTALL)
        if m:
            value = re.sub(r"<.*?>", "", m.group(1)).strip()
            return value if value else None
    return None

### Geocoding + nearest/count functions

In [54]:
def geocode_address_onemap(address, session, max_retries=5, base_sleep=1.5):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": address,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    for attempt in range(max_retries):
        try:
            r = session.get(url, params=params, timeout=20)
            r.raise_for_status()
            data = r.json()

            if data.get("found", 0) > 0 and len(data.get("results", [])) > 0:
                first = data["results"][0]
                return {
                    "address": address,
                    "latitude": float(first["LATITUDE"]),
                    "longitude": float(first["LONGITUDE"]),
                    "matched_address": first.get("ADDRESS"),
                    "status": "ok"
                }
            else:
                return {
                    "address": address,
                    "latitude": np.nan,
                    "longitude": np.nan,
                    "matched_address": None,
                    "status": "not_found"
                }

        except Exception as e:
            wait = base_sleep * (2 ** attempt)
            print(f"Retry {attempt+1}/{max_retries} failed for: {address} | {type(e).__name__}: {e}")
            time.sleep(wait)

    return {
        "address": address,
        "latitude": np.nan,
        "longitude": np.nan,
        "matched_address": None,
        "status": "failed"
    }

def add_nearest(points_gdf, layer_gdf, name_col, prefix, lat_col, lon_col):
    points_sv = project_to_sv21(points_gdf[[lat_col, lon_col, "geometry"]].copy())
    layer_sv = project_to_sv21(layer_gdf[[name_col, "geometry"]].dropna(subset=["geometry"]).copy())

    joined = gpd.sjoin_nearest(
        points_sv,
        layer_sv,
        how="left",
        distance_col=f"nearest_{prefix}_distance_m"
    )

    out = joined[[lat_col, lon_col, name_col, f"nearest_{prefix}_distance_m"]].copy()
    out = out.rename(columns={name_col: f"nearest_{prefix}_name"})
    out[f"nearest_{prefix}_distance_m"] = out[f"nearest_{prefix}_distance_m"].round(1)

    return out.drop_duplicates(subset=[lat_col, lon_col])

def add_count_within_radius(
    points_gdf,
    layer_gdf,
    prefix,
    lat_col,
    lon_col,
    radius_m=1000,
    unique_by=None
):
    layer = layer_gdf.dropna(subset=["geometry"]).copy()
    layer = project_to_sv21(layer)

    if unique_by is not None:
        layer = layer[[unique_by, "geometry"]].dropna(subset=[unique_by]).copy()
        count_key = unique_by
    else:
        count_key = "__feature_id"
        layer[count_key] = layer.index.astype(str)

    buffers = points_gdf[[lat_col, lon_col, "geometry"]].copy()
    buffers = project_to_sv21(buffers)
    buffers["geometry"] = buffers.geometry.buffer(radius_m)

    joined = gpd.sjoin(
        buffers,
        layer[[count_key, "geometry"]],
        how="left",
        predicate="intersects"
    )

    joined = joined.dropna(subset=[count_key]).copy()

    counts = (
        joined[[lat_col, lon_col, count_key]]
        .drop_duplicates()
        .groupby([lat_col, lon_col])[count_key]
        .nunique()
        .reset_index(name=f"num_{prefix}_within_{int(radius_m)}m")
    )

    out = points_gdf[[lat_col, lon_col]].drop_duplicates().copy()
    out = out.merge(counts, on=[lat_col, lon_col], how="left")
    out[f"num_{prefix}_within_{int(radius_m)}m"] = (
        out[f"num_{prefix}_within_{int(radius_m)}m"].fillna(0).astype(int)
    )
    return out

### Layer builders

In [55]:
def build_mrt_layer(mrt_file):
    mrt_gdf = gpd.read_file(mrt_file)
    mrt_gdf = normalize_to_wgs84(mrt_gdf)

    mrt_gdf["lon"] = mrt_gdf.geometry.x
    mrt_gdf["lat"] = mrt_gdf.geometry.y

    mrt_station = mrt_gdf.groupby("STATION_NA", as_index=False)[["lat", "lon"]].mean()

    return gpd.GeoDataFrame(
        mrt_station.rename(columns={"STATION_NA": "mrt_name"}),
        geometry=gpd.points_from_xy(mrt_station["lon"], mrt_station["lat"]),
        crs="EPSG:4326"
    )

def build_clinic_layer(amenity_dir):
    clinic_file = find_file(amenity_dir, ["chas"])
    clinics = gpd.read_file(clinic_file)
    clinics = normalize_to_wgs84(clinics)

    if "Description" in clinics.columns:
        clinics["clinic_name"] = clinics["Description"].apply(
            lambda x: extract_from_description(x, "HCI_NAME")
        )
        clinics["hci_code"] = clinics["Description"].apply(
            lambda x: extract_from_description(x, "HCI_CODE")
        )
    else:
        if "NAME" in clinics.columns:
            clinics["clinic_name"] = clinics["NAME"]
        else:
            clinics["clinic_name"] = "Clinic"

        clinics["hci_code"] = pd.Series(clinics.index.astype(str), index=clinics.index)

    clinics["clinic_name"] = clinics["clinic_name"].fillna("Clinic")
    clinics["hci_code"] = clinics["hci_code"].fillna(
        pd.Series(clinics.index.astype(str), index=clinics.index)
    )

    return clinics[["clinic_name", "hci_code", "geometry"]].dropna(subset=["geometry"]).copy()

def build_parks_layer(amenity_dir):
    park_file = find_file(amenity_dir, ["park"])
    parks = gpd.read_file(park_file)
    parks = normalize_to_wgs84(parks)

    if "NAME" in parks.columns:
        parks["park_name"] = parks["NAME"]
    elif "name" in parks.columns:
        parks["park_name"] = parks["name"]
    else:
        parks["park_name"] = "Park"

    return parks[["park_name", "geometry"]].dropna(subset=["geometry"]).copy()

def build_community_club_layer(amenity_dir):
    cc_file = find_file(amenity_dir, ["community"])
    ccs = gpd.read_file(cc_file)
    ccs = normalize_to_wgs84(ccs)

    if "NAME" in ccs.columns:
        ccs["community_club_name"] = ccs["NAME"]
    elif "name" in ccs.columns:
        ccs["community_club_name"] = ccs["name"]
    else:
        ccs["community_club_name"] = "Community Club"

    return ccs[["community_club_name", "geometry"]].dropna(subset=["geometry"]).copy()

def build_hawker_layer(amenity_dir):
    hawker_file = find_file(amenity_dir, ["hawker"])
    hawkers = gpd.read_file(hawker_file)
    hawkers = normalize_to_wgs84(hawkers)

    if "NAME" in hawkers.columns:
        hawkers["hawker_name"] = hawkers["NAME"]
    elif "Name" in hawkers.columns:
        hawkers["hawker_name"] = hawkers["Name"]
    else:
        hawkers["hawker_name"] = "Hawker Centre"

    if "STATUS" in hawkers.columns:
        hawkers = hawkers[hawkers["STATUS"].fillna("").str.lower() == "existing"].copy()

    return hawkers[["hawker_name", "geometry"]].dropna(subset=["geometry"]).copy()

### Prepare final_df and geocode addresses

In [56]:
output_df = hdb_df_final.copy()

if "address" not in output_df.columns:
    if {"block", "street_name"}.issubset(output_df.columns):
        output_df["block"] = output_df["block"].astype(str).str.replace(r"\.0+$", "", regex=True).str.strip()
        output_df["street_name"] = output_df["street_name"].astype(str).str.strip()
        output_df["address"] = (output_df["block"] + " " + output_df["street_name"]).str.strip()
    else:
        raise KeyError("Need either an 'address' column or both 'block' and 'street_name' columns.")

output_df["address"] = output_df["address"].astype(str).str.strip()
output_df.loc[output_df["address"].isin(["", "nan", "None"]), "address"] = np.nan

unique_addresses = pd.Series(output_df["address"].dropna().unique(), name="address")
log(f"Unique addresses to geocode: {len(unique_addresses)}")

if Path(CACHE_FILE).exists():
    cache_df = pd.read_csv(CACHE_FILE)

    # make old cache compatible
    if "address" not in cache_df.columns and "full_address" in cache_df.columns:
        cache_df = cache_df.rename(columns={"full_address": "address"})

    required_cols = ["address", "latitude", "longitude", "matched_address", "status"]
    for col in required_cols:
        if col not in cache_df.columns:
            cache_df[col] = np.nan

    cache_df = cache_df[required_cols].copy()
    done_addresses = set(cache_df["address"].astype(str))
    log(f"Loaded cache rows: {len(cache_df)}")
else:
    cache_df = pd.DataFrame(columns=["address", "latitude", "longitude", "matched_address", "status"])
    done_addresses = set()

to_geocode = [addr for addr in unique_addresses if str(addr) not in done_addresses]
log(f"Remaining addresses to geocode: {len(to_geocode)}")

results = []

with requests.Session() as session:
    for i, address in enumerate(to_geocode, start=1):
        result = geocode_address_onemap(address, session=session)
        results.append(result)
        time.sleep(GEOCODE_SLEEP)

        if i % SAVE_EVERY == 0:
            batch_df = pd.DataFrame(results)
            cache_df = pd.concat([cache_df, batch_df], ignore_index=True)
            cache_df = cache_df.drop_duplicates(subset=["address"], keep="last")
            cache_df.to_csv(CACHE_FILE, index=False)
            log(f"Saved geocode progress at {i}/{len(to_geocode)}")
            results = []

if results:
    batch_df = pd.DataFrame(results)
    cache_df = pd.concat([cache_df, batch_df], ignore_index=True)
    cache_df = cache_df.drop_duplicates(subset=["address"], keep="last")
    cache_df.to_csv(CACHE_FILE, index=False)

log(f"Final cache rows: {len(cache_df)}")

addr_lookup = pd.DataFrame({"address": unique_addresses})
addr_lookup = addr_lookup.merge(
    cache_df[["address", "latitude", "longitude", "matched_address", "status"]],
    on="address",
    how="left"
)

points_gdf = build_points_gdf(addr_lookup, lat_col="latitude", lon_col="longitude")
log(f"Addresses with valid coordinates: {len(points_gdf)}")

[INFO] Unique addresses to geocode: 9698
[INFO] Loaded cache rows: 9698
[INFO] Remaining addresses to geocode: 0
[INFO] Final cache rows: 9698
[INFO] Addresses with valid coordinates: 9698


### Load layers + compute nearest + counts

In [57]:
log("Loading MRT / amenity layers...")
mrt = build_mrt_layer(MRT_FILE)
clinics = build_clinic_layer(AMENITY_DIR)
parks = build_parks_layer(AMENITY_DIR)
community_clubs = build_community_club_layer(AMENITY_DIR)
hawkers = build_hawker_layer(AMENITY_DIR)

log("Computing nearest MRT / amenities...")
nearest_mrt = add_nearest(points_gdf, mrt, "mrt_name", "mrt", "latitude", "longitude")
nearest_clinic = add_nearest(points_gdf, clinics, "clinic_name", "clinic", "latitude", "longitude")
nearest_park = add_nearest(points_gdf, parks, "park_name", "park", "latitude", "longitude")
nearest_cc = add_nearest(points_gdf, community_clubs, "community_club_name", "community_club", "latitude", "longitude")
nearest_hawker = add_nearest(points_gdf, hawkers, "hawker_name", "hawker", "latitude", "longitude")

log("Computing counts within 1km...")
mrt_counts = add_count_within_radius(
    points_gdf, mrt, "mrt", "latitude", "longitude",
    radius_m=RADIUS_M, unique_by="mrt_name"
)
clinic_counts = add_count_within_radius(
    points_gdf, clinics, "clinic", "latitude", "longitude",
    radius_m=RADIUS_M, unique_by="hci_code"
)
park_counts = add_count_within_radius(
    points_gdf, parks, "park", "latitude", "longitude",
    radius_m=RADIUS_M, unique_by="park_name"
)
cc_counts = add_count_within_radius(
    points_gdf, community_clubs, "community_club", "latitude", "longitude",
    radius_m=RADIUS_M, unique_by="community_club_name"
)
hawker_counts = add_count_within_radius(
    points_gdf, hawkers, "hawker", "latitude", "longitude",
    radius_m=RADIUS_M, unique_by="hawker_name"
)

[INFO] Loading MRT / amenity layers...
[INFO] Computing nearest MRT / amenities...
[INFO] Computing counts within 1km...


### Merge features back into final_df

In [58]:
feature_lookup = addr_lookup.copy()

for feat_df in [
    nearest_mrt,
    nearest_clinic,
    nearest_park,
    nearest_cc,
    nearest_hawker,
    mrt_counts,
    clinic_counts,
    park_counts,
    cc_counts,
    hawker_counts
]:
    feature_lookup = feature_lookup.merge(
        feat_df,
        on=["latitude", "longitude"],
        how="left"
    )

feature_lookup["num_amenities_within_1000m"] = (
    feature_lookup["num_clinic_within_1000m"].fillna(0)
    + feature_lookup["num_park_within_1000m"].fillna(0)
    + feature_lookup["num_community_club_within_1000m"].fillna(0)
    + feature_lookup["num_hawker_within_1000m"].fillna(0)
).astype(int)

feature_lookup = feature_lookup.drop_duplicates(subset=["address"])

drop_cols = [
    "latitude", "longitude", "matched_address", "status",
    "nearest_mrt_name", "nearest_mrt_distance_m",
    "nearest_clinic_name", "nearest_clinic_distance_m",
    "nearest_park_name", "nearest_park_distance_m",
    "nearest_community_club_name", "nearest_community_club_distance_m",
    "nearest_hawker_name", "nearest_hawker_distance_m",
    "num_mrt_within_1000m",
    "num_clinic_within_1000m",
    "num_park_within_1000m",
    "num_community_club_within_1000m",
    "num_hawker_within_1000m",
    "num_amenities_within_1000m",
]

output_df = output_df.drop(columns=[c for c in drop_cols if c in output_df.columns], errors="ignore")

output_df = output_df.merge(
    feature_lookup[[
        "address",
        "latitude", "longitude", "matched_address", "status",
        "nearest_mrt_name", "nearest_mrt_distance_m",
        "nearest_clinic_name", "nearest_clinic_distance_m",
        "nearest_park_name", "nearest_park_distance_m",
        "nearest_community_club_name", "nearest_community_club_distance_m",
        "nearest_hawker_name", "nearest_hawker_distance_m",
        "num_mrt_within_1000m",
        "num_clinic_within_1000m",
        "num_park_within_1000m",
        "num_community_club_within_1000m",
        "num_hawker_within_1000m",
        "num_amenities_within_1000m"
    ]],
    on="address",
    how="left"
)

final_output_df = output_df.copy()

### Save and preview

In [59]:
final_output_df.to_csv('HDB_full_resale_info.csv.gz', compression='gzip', index=False)